# 00 - Refresh Environment / Verify Lakebase Connectivity

Validates that this notebook can authenticate to the Lakebase Postgres
instance via `WorkspaceClient` and generate a fresh database credential.

**No `.env` file needed** - credentials come from the notebook execution
context running in the same workspace as the Lakebase instance.

## Configuration

Set the widget values or edit the defaults below to match your Lakebase instance.

In [ ]:
dbutils.widgets.text("pg_project_id", "nominatim-lakebase", "Lakebase Project ID")
dbutils.widgets.text("pg_branch_id", "production", "Lakebase Branch ID")
dbutils.widgets.text("pg_user", "justin.monaldo@databricks.com", "PG User")
dbutils.widgets.text("pg_database", "nominatim", "PG Database")
dbutils.widgets.text("pg_port", "5432", "PG Port")

In [ ]:
pg_project_id = dbutils.widgets.get("pg_project_id")
pg_branch_id = dbutils.widgets.get("pg_branch_id")
pg_user = dbutils.widgets.get("pg_user")
pg_database = dbutils.widgets.get("pg_database")
pg_port = dbutils.widgets.get("pg_port")

print("=" * 60)
print("Lakebase Connectivity Check")
print("=" * 60)
print(f"  Project:  {pg_project_id}")
print(f"  Branch:   {pg_branch_id}")
print(f"  User:     {pg_user}")
print(f"  Database: {pg_database}")
print(f"  Port:     {pg_port}")
print()

## Generate Fresh Token via WorkspaceClient

In [ ]:
%run ./_helpers

In [ ]:
w = get_workspace_client()
token = get_lakebase_token(pg_project_id, pg_branch_id, w=w)
print(f"Token generated successfully (length={len(token)} chars)")
print(f"Token prefix: {token[:20]}...")
print()
print("The token is valid for ~1 hour.")

## Verify Database Connection

In [ ]:
import psycopg2

pg_env = get_pg_env(
    project_id=pg_project_id,
    branch_id=pg_branch_id,
    user=pg_user,
    database="postgres",  # connect to maintenance DB first
    port=pg_port,
)

conn_params = get_psycopg_conn_params(pg_env)
conn = psycopg2.connect(**conn_params)
conn.autocommit = True
cur = conn.cursor()
cur.execute("SELECT version();")
version = cur.fetchone()[0]
print(f"Connected successfully!")
print(f"PostgreSQL version: {version}")
cur.close()
conn.close()

In [ ]:
print("=" * 60)
print("Environment check PASSED")
print("=" * 60)
print()
print("You can proceed with the remaining notebooks:")
print("  01_setup_postgis  -> Install PostGIS & hstore extensions")
print("  02_download_osm_data -> Download OSM .pbf files")
print("  03_build_nominatim_server -> Import data into Nominatim")